In [1]:
   !pip install -q langchain langchain-community langchain-text-splitters langchain-huggingface langchain-chroma chromadb pypdf sentence-transformers langgraph

In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("IT-Policy.pdf")
pages = loader.load()

print("Number of pages:", len(pages))
print(pages[0].page_content[:500])

/tmp/ipykernel_94784/4017325536.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Number of pages: 22
TWOC IT Policies 
 
 
Page 1 of 22 
 
 
Information Technology Policy 
 
 
 
And 
 
 
 
Procedure Manual 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Tide Water Oil Co. (India) Ltd 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
This document contains confidential information and remains the property of Tide Water Oil Co. (India) Ltd. . This 
manual is intended to assist our staff in day-to-day operations and rendering services to our clients/customers. It is not 
to be used for any other purposes, copied, dis


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,     # max characters per chunk
    chunk_overlap=200    # characters repeated between neighbouring chunks
)

chunks = splitter.split_documents(pages)

print("Number of chunks:", len(chunks))
print("---- Chunk 10 ----")
print(chunks[10].page_content)
print("Metadata:", chunks[10].metadata)

Number of chunks: 76
---- Chunk 10 ----
4. Applicability of the Policy 
 
The IT Policy is designed by the Information Technology Department and approved by the senior 
management, and as such, will apply to: 
 
➢ All employees of TWOC Industries Ltd and its subsidiaries, sales offices, affiliates and users of IT 
facilities within TWOC. 
➢ All contractors / service providers of TWOC making use of IT facilities 
➢ All employees of the contractors of ITD, to the extent that they are involved in carrying out day-
to-day activities for the Company. 
➢ All systems used by ITD (i.e. computer systems, networks, ancillary equipment, storage media 
etc.). 
➢ Any other area defined by the CIO in a formal amendment to the IT Policy.
Metadata: {'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2022-03-31T12:19:06+05:30', 'author': 'TAPAS_SAHA', 'moddate': '2022-03-31T12:19:06+05:30', 'source': 'IT-Policy.pdf', 'total_pages': 22, 'pag

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

test = embeddings.embed_query("What is the password policy?")
print("Length of the embedding:", len(test))
print("First 5 numbers:", test[:5])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Length of the embedding: 384
First 5 numbers: [-0.027855973690748215, -0.008416321128606796, -0.07506153732538223, -0.05250142887234688, -0.009212225675582886]


In [5]:
from langchain_chroma import Chroma

vectorstore = Chroma(
    collection_name="it_policy",
    embedding_function=embeddings,
    persist_directory="chroma_db"
)
vectorstore.reset_collection()      # start empty on every run, so no duplicates
vectorstore.add_documents(chunks)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("Stored chunks:", vectorstore._collection.count())   # should print 76

Stored chunks: 76


In [6]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def generate_text(prompt):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    output_ids = model.generate(**inputs, max_new_tokens=150)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [7]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, END

THRESHOLD = 1.3

class RAGState(TypedDict):
    question: str
    context: str
    pages: List[str]
    best_score: float
    answer: str
    escalated: bool

def retrieve(state: RAGState):
    results = vectorstore.similarity_search_with_score(state["question"], k=3)
    context = "\n\n".join(doc.page_content for doc, _ in results)
    pages = sorted({doc.metadata["page_label"] for doc, _ in results})
    best = min(score for _, score in results)
    return {"context": context, "pages": pages, "best_score": best}

def generate(state: RAGState):
    prompt = (
        "Answer the question using only the context below. "
        "If the answer is not in the context, say \"I don't know\".\n\n"
        f"Question: {state['question']}\n\n"
        f"Context:\n{state['context'][:1500]}\n\n"
        "Answer:"
    )
    return {"answer": generate_text(prompt), "escalated": False}

def escalate(state: RAGState):
    return {
        "answer": "I'm not confident I can answer this from the policy document. "
                  "I have forwarded your question to a human agent.",
        "escalated": True,
    }

def route(state: RAGState):
    return "escalate" if state["best_score"] > THRESHOLD else "generate"

graph = StateGraph(RAGState)
graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate)
graph.add_node("escalate", escalate)

graph.set_entry_point("retrieve")
graph.add_conditional_edges("retrieve", route, {"generate": "generate", "escalate": "escalate"})
graph.add_edge("generate", END)
graph.add_edge("escalate", END)

app = graph.compile()

In [8]:
def ask(question):
    result = app.invoke({"question": question})
    print("Question  :", question)
    print("Best score:", round(result["best_score"], 3))
    print("Escalated :", result["escalated"])
    print("Answer    :", result["answer"])
    print("Sources   : pages", result["pages"])
    print()

# Questions the document can answer
ask("Who does this policy apply to?")
ask("What is the password policy?")

# A question the document cannot answer: should escalate to a human
ask("What is the capital of France?")

# Known limitation: the search matches the word "leave", not its meaning
ask("What is the leave policy for employees?")

Question  : Who does this policy apply to?
Best score: 0.986
Escalated : False
Answer    : All employees of TWOC Industries Ltd and its subsidiaries, sales offices, affiliates and users of IT facilities within TWOC.  All contractors / service providers of TWOC making use of IT facilities  All employees of the contractors of ITD, to the extent that they are involved in carrying out day- to-day activities for the Company.  All systems used by ITD (i.e. computer systems, networks, ancillary equipment, storage media etc.).  Any other area defined by the CIO in a formal amendment to the IT Policy.
Sources   : pages ['2', '4']

Question  : What is the password policy?
Best score: 0.966
Escalated : False
Answer    : Passwords would not be stored or transmitted in a readable form (i.e. plain text form) as far as possible.  Passwords would not be hard coded into software as far as practical and possible.  Initial passwords given to users shall be changed immediately by the user himself.  Initia